# Session 5 — Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tech4alltraining/aiml/blob/main/mlai-genai-internship/student/notebooks/session-05-regression.ipynb)

**ML/AI & GenAI Internship** · Linear Regression · MAE, MSE, RMSE, R² · Three complete use cases

---

## How to use this notebook

**This is where you train your first model.** Regression predicts a **number**.

| | Dataset | What it teaches |
|---|---|---|
| **Use case 1** | `salary_data.csv` | The whole workflow, and **what every metric means** |
| **Use case 2** | `advertising.csv` | Reading coefficients; dropping a useless feature |
| **Use case 3** | `cardekho_dataset.csv` | Encoding, scaling, and what to do when the model is not good enough |

Run the cells in order — each builds on the last.

**20 MCQs and 15 tasks:** [Session 5 guide](../sessions/session-05-regression.md).

## How this session is organised

**Three complete use cases**, each chosen to show something the previous one could not.

| | Dataset | Features | What it teaches |
|---|---|---|---|
| **Use case 1** | `salary_data.csv` | **1** — years of experience | The whole workflow, and **what every metric means** |
| **Use case 2** | `advertising.csv` | **3** — TV, radio, newspaper | Reading coefficients, and dropping a useless feature |
| **Use case 3** | `cardekho_dataset.csv` | **10** — a real used-car market | Encoding, scaling, and **what to do when the model is not good enough** |

| # | Topic |
|---|---|
| 1 | [Supervised learning: regression and classification](#1-supervised-learning-regression-and-classification) |
| 2 | [Use case 1 — Salary from experience](#2-use-case-1--salary-from-experience) |
| 3 | [Use case 2 — Advertising spend and sales](#3-use-case-2--advertising-spend-and-sales) |
| 4 | [Use case 3 — Used car prices](#4-use-case-3--used-car-prices) |
| 5 | [What the three use cases showed](#5-what-the-three-use-cases-showed) |
| | [❓ 20 MCQs](#-regression--20-mcqs) · [🎯 Tasks](#-regression--tasks) |

**Practices sit between the use cases.** The MCQs and tasks are at the end.

# 1. Supervised learning: regression and classification

**Supervised** means you have the answers. You show the model inputs *and* correct outputs, and it learns the mapping between them.

🧠 **Analogy: past exam papers with the answer key.** You study a hundred solved problems. Nobody explained the underlying theory — you inferred it from worked examples. Then you sit a new paper. **The final exam is the test set.**

## The two kinds

| | Regression | Classification |
|---|---|---|
| The answer is | A **number** | A **category** |
| Question | *How much? How many?* | *Which one? Yes or no?* |
| Example | Salary from experience | Loan approved or rejected |
| A bad prediction is | Off by ₹4,000 | Simply wrong |
| Typical metric | RMSE, R² | Accuracy, F1 |

> **A regression prediction can be nearly right. A classification prediction is right or wrong.** That single difference is why the two families need completely different metrics — and it is why this session is split into Part A and Part B.

## The trap: numbers that are really categories

```text
age 30 + age 30 = 60        -> meaningful  -> age is a NUMBER
pincode + pincode           -> nonsense    -> pincode is a CATEGORY
```

**If the arithmetic is meaningless, it is a category** — however it is stored. Phone numbers, roll numbers, postcodes and any ID column all look numeric and are not.

# 2. Use case 1 — Salary from experience

**One feature, one target, and no complications.** That is exactly why we start here: **every step is visible, and you can concentrate on what the metrics mean.**

**The question:** given someone's years of experience, predict their salary.

## Step 1 — Import the libraries and load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

dataset_url = "https://raw.githubusercontent.com/tech4alltraining/aiml/refs/heads/main/datasets/regression/salary_data.csv"

df = pd.read_csv(dataset_url)
df.head()

**Output:**

```text
   Experience    Salary
0         5.0   90000.0
1         3.0   65000.0
2        15.0  150000.0
3         7.0   60000.0
4        20.0  200000.0
```

**Two columns.** `Experience` is the **feature** — the information we are allowed to use. `Salary` is the **target** — the number we must predict.

## Step 2 — Exploratory Data Analysis

**Exactly the routine from Session 3.** Look before you touch anything.

In [ ]:
df.head()
df.tail()
df.info()
df.describe()

**Output of `info()`:**

```text
RangeIndex: 375 entries, 0 to 374
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Experience  373 non-null    float64
 1   Salary      373 non-null    float64
```

**375 rows, but only 373 non-null in each column — so there are two gaps in each.**

**Output of `describe()`:**

```text
       Experience         Salary
count  373.000000     373.000000
mean    10.030831  100577.345845
std      6.557007   48240.013482
min      0.000000     350.000000
25%      4.000000   55000.000000
50%      9.000000   95000.000000
75%     15.000000  140000.000000
max     25.000000  250000.000000
```

**Experience runs 0 to 25 years; salary runs 350 to 250,000.** The mean and median salary are close (100,577 against 95,000), so this column is **not badly skewed** — unlike the salary column you met in Session 3.

### Correlation — does experience actually relate to salary?

**Before building anything, check that there is a relationship to find.**

In [ ]:
df.corr()

**Output:**

```text
            Experience    Salary
Experience    1.000000  0.930338
Salary        0.930338  1.000000
```

**0.93 is a very strong positive correlation.** As experience rises, salary rises with it, closely. **This is the number that tells you a linear model is worth trying at all.**

In [ ]:
sns.heatmap(df.corr(), annot=True)
plt.show()

**A heatmap is overkill for two columns**, but on a dataset with fifteen it is the fastest way to see which pairs move together. **Learn the habit here where it is easy to read.**

> **If the correlation had been 0.05, a straight line would have been the wrong tool** — and you would have found that out in ten seconds rather than after building a model.

## Step 3 — Preprocessing

### Missing values

In [ ]:
df.isnull().sum()

**Output:**

```text
Experience    2
Salary        2
```

**Two gaps in each column.** With 375 rows, dropping them costs well under 1% of the data — **so dropping is the sensible choice here.**

In [ ]:
df.dropna(inplace=True)
print(df.shape)

**Output:**

```text
(373, 2)
```

> **Compare this with Session 3's `pre_data.csv`**, where dropping three rows cost 25% of the dataset and imputation was the right answer. **Same technique, opposite decision — and the deciding factor is how much data you have.**

### Duplicates

In [ ]:
df.duplicated().sum()

**Output:**

```text
219
```

**219 duplicates out of 373 rows — that is 59% of the dataset.**

> ⚠️ **Do not remove these.**
>
> **Look at what the columns are.** `Experience` is a whole number of years from 0 to 25. `Salary` is a round figure. **With only 26 possible experience values and a few dozen common salary figures, two different people having identical rows is completely ordinary.**
>
> **Two people with 5 years of experience both earning 90,000 is not a data-entry error. It is two people.**
>
> Removing them would delete 59% of a legitimate dataset — and it would systematically delete the *most common* combinations, which are exactly the ones the model most needs to learn.

**This is Session 3's lesson made concrete:** *before dropping duplicates, ask whether two rows could legitimately be identical.* **Here, obviously yes.**

In [ ]:
# df.drop_duplicates(inplace=True)   <- deliberately NOT done

### Outliers

In [ ]:
plt.boxplot(df['Experience'])
plt.show()

plt.boxplot(df['Salary'])
plt.show()

**Neither box plot shows any dots beyond the whiskers.** The IQR bounds work out at −12 to 32 years and −72,500 to 267,500 — **and every value falls comfortably inside.**

**No outliers to handle.**

### Encoding and scaling

**Neither is needed here:**

- **Encoding** converts text to numbers. **Both columns are already numeric**, so there is nothing to encode.
- **Scaling** puts columns on a comparable range. **Linear regression does not require it** — and with a single feature there is nothing to compare against anyway.

> **Recognising that a step is unnecessary is as important as knowing how to do it.** A pipeline that scales a column for no reason is not more careful; it is just longer.

## Step 4 — Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Salary', axis=1)      # features - everything except the target
y = df['Salary']                   # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(df.shape, X.shape, y.shape)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

**Output:**

```text
(373, 2) (373, 1) (373,)
(298, 1) (298,)
(75, 1) (75,)
```

**Read those shapes carefully — they tell you the split worked.**

| | Shape | Meaning |
|---|---|---|
| `X` | `(373, 1)` | 373 rows, **1 feature** |
| `y` | `(373,)` | 373 targets, and no second dimension |
| `X_train` | `(298, 1)` | 80% of the rows |
| `X_test` | `(75, 1)` | the remaining 20% |

**298 + 75 = 373.** Nothing was lost.

> **`X` is a DataFrame (two-dimensional) and `y` is a Series (one-dimensional).** That is not an accident — scikit-learn expects a table of features and a single column of answers. **`df.drop('Salary', axis=1)` keeps `X` two-dimensional even with one feature**, which is what the model needs.

> **No `stratify` here.** That argument keeps class proportions balanced, and a regression target has no classes to balance. **It is for classification only.**

## Step 5 — Model selection and training

**Linear regression fits a straight line through the data.**

```text
Salary = slope × Experience + intercept
```

**"Training" means finding the slope and intercept that make the errors smallest.**

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

**Two lines.** `LinearRegression()` creates the model; `.fit()` learns from the training data.

> **This is the pattern from Session 4's Topic 19** — create, train, predict, measure. **Every model in scikit-learn works this way**, so learning it once is enough.

## Step 6 — Making predictions

In [ ]:
y_pred = model.predict(X_test)
y_pred[:5]

**Output:**

```text
[174795.47,  99746.98, 140682.52,  72456.62, 147505.11]
```

In [ ]:
y_test[:5]

**Output:**

```text
[180000.0, 65000.0, 125000.0, 80000.0, 140000.0]
```

**Now compare them, row by row:**

| | Actual | Predicted | Difference |
|---|---|---|---|
| 1 | 180,000 | 174,795 | **−5,205** |
| 2 | 65,000 | 99,747 | **+34,747** |
| 3 | 125,000 | 140,683 | +15,683 |
| 4 | 80,000 | 72,457 | −7,543 |
| 5 | 140,000 | 147,505 | +7,505 |

**Some predictions are close; one is out by nearly 35,000.** **The whole point of the next step is to turn this column of differences into a single number you can report.**

## Step 7 — Evaluation metrics, properly

**This is the most important section in Part A.** Four metrics, and each answers a different question.

### The problem with just adding up the errors

**Take four predictions with these errors:**

```text
prediction 1:  actual 10, predicted 11  ->  error = 10 - 11 = -1
prediction 2:  actual 15, predicted 14  ->  error = 15 - 14 = +1
prediction 3:  actual  9, predicted 11  ->  error =  9 - 11 = -2
prediction 4:  actual  8, predicted  6  ->  error =  8 -  6 = +2
```

**Add them up:**

```text
-1 + 1 + -2 + 2  =  0
```

> **Total error zero — and every single prediction was wrong.**
>
> **The positive and negative errors cancelled out.** This is why you can never simply sum the errors, and it is the reason all four metrics below do something to remove the sign.

**There are two ways to remove a sign: take the absolute value, or square it. Those two choices give you MAE and MSE.**

---

### MAE — Mean Absolute Error

**Take the absolute value of each error, then average them.**

```text
|-1| + |+1| + |-2| + |+2|  =  1 + 1 + 2 + 2  =  6
MAE = 6 / 4 = 1.5
```

**In words: on average, this model is wrong by 1.5.**

| | |
|---|---|
| **Units** | The same as your target |
| **Reads as** | "typically off by about this much" |
| **Treats all errors** | Equally — an error of 10 counts ten times an error of 1 |

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
print(f'Mean Absolute Error: {mae}')

**Output:**

```text
Mean Absolute Error: 12094.170266826043
```

> **Read that out loud: "the model is typically wrong by about ₹12,094".** **That sentence is what a non-technical person actually needs**, and MAE is the metric that gives it to you most directly.

---

### MSE — Mean Squared Error

**Square each error, then average them.**

```text
(-1)² + (+1)² + (-2)² + (+2)²  =  1 + 1 + 4 + 4  =  10
MSE = 10 / 4 = 2.5
```

**Squaring removes the sign too — but it does something else as well.**

> **Squaring punishes large errors far more than small ones.** An error of 10 becomes 100; an error of 1 becomes 1. **One big miss now counts a hundred times a small one.**
>
> **That is sometimes exactly what you want** — if being badly wrong occasionally is worse than being slightly wrong often.

In [ ]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

**Output:**

```text
Mean Squared Error: 241834883.89985102
```

> ⚠️ **241 million?** **The units are rupees *squared*.** Nobody can interpret that, and you should never put it in a report for a human. **MSE is useful for comparing two models, not for describing one.**

---

### RMSE — Root Mean Squared Error

**Take the square root of MSE.**

```text
RMSE = √2.5 = 1.58
```

**This undoes the squaring, so the number lands back in your target's units** — while keeping MSE's heavier punishment of large errors.

In [ ]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_test, y_pred)
print(f'Root Mean Squared Error: {rmse}')

**Output:**

```text
Root Mean Squared Error: 15551.041184437
```

> **RMSE is the metric to quote.** It is in rupees, it is interpretable, and it does not hide occasional large mistakes the way MAE can.

### MAE and RMSE together tell you something neither says alone

```text
MAE  = 12,094
RMSE = 15,551
```

**RMSE is always at least as large as MAE.** **How much larger tells you about the shape of your errors:**

| | Means |
|---|---|
| **RMSE ≈ MAE** | Errors are all roughly the same size |
| **RMSE ≫ MAE** | A few predictions are badly wrong, and dragging RMSE up |

**Here RMSE is about 29% above MAE — a moderate gap.** Most predictions are decent, and a handful are noticeably worse. **Looking at the two together is free information most people ignore.**

---

### R² — the coefficient of determination

**The first three metrics tell you *how wrong* you are. R² tells you *how much better than nothing* you are.**

🧠 **Analogy: comparing against a lazy guess.** Suppose you refuse to build a model and simply predict the *average salary* for everyone. That is the worst reasonable guess. **R² asks: how much of the error did the model remove, compared with that lazy guess?**

| R² | Meaning |
|---|---|
| **1.0** | Perfect — every prediction exact |
| **0.9** | The model explains 90% of the variation |
| **0.0** | No better than just guessing the average |
| **Negative** | **Worse than guessing the average** — yes, this is possible |

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print(f'R-squared: {r2}')

**Output:**

```text
R-squared: 0.8991335611
```

> **R² of 0.899 means the model explains about 90% of the variation in salary.** For a model with a single feature, that is a genuinely good result — **and it is what the 0.93 correlation at step 2 predicted.**

> ⚠️ **R² of 1.0 on a test set is not a triumph — it is a warning.** Real data has noise, so a perfect score almost always means the target leaked into your features.

---

### The four metrics side by side

| Metric | Value | Units | Use it to |
|---|---|---|---|
| **MAE** | 12,094 | Rupees | Say "typically off by about ₹12,000" |
| **MSE** | 241,834,884 | Rupees **squared** | Compare two models — never to describe one |
| **RMSE** | 15,551 | Rupees | **Report this one** |
| **R²** | 0.899 | None | Say "explains 90% of the variation" |

> **Report RMSE and R² together.** RMSE says how wrong you are in real units; R² says whether that is good relative to doing nothing. **Neither is enough alone.**

## Step 8 — Reading the model

**A linear model can be read out loud, and this is one of its great advantages.**

In [ ]:
slope = model.coef_
intercept = model.intercept_

print(f'Slope: {slope}')
print(f'Y-Intercept: {intercept}')
print(f'X-Intercept: {-intercept/slope}')

**Output:**

```text
Slope: [6822.59017499]
Y-Intercept: 31521.077629
X-Intercept: [-4.62012]
```

**The learned equation is:**

```text
Salary = 6,822.59 × Experience + 31,521.08
```

| Term | Value | What it means |
|---|---|---|
| **Slope** | 6,822.59 | **Each extra year of experience is worth about ₹6,823** |
| **Y-intercept** | 31,521.08 | The predicted salary at zero experience — a starting salary |
| **X-intercept** | −4.62 | Where the line crosses zero salary |

> **The x-intercept of −4.62 years is meaningless**, and worth saying so. It is where the mathematics puts the line, not a fact about the world — **you cannot have −4.62 years of experience.** **A model can be extended beyond its data; its answers there should not be trusted.**

> **"Each extra year of experience is worth about ₹6,823, starting from around ₹31,500."** **A model you can say in one sentence is a model you can defend in a meeting** — and no other family in this session gives you that as easily.

## Step 9 — Plotting the result

In [ ]:
plt.scatter(X_test, y_test, color='black', label='Actual data')
plt.plot(X_test, y_pred, color='blue', linewidth=1,
         label='Regression line', marker='*')
plt.xlabel('Experience')
plt.ylabel('Salary')
plt.title('Linear Regression on Salary Data')
plt.legend()
plt.show()

![Scatter of actual salaries with the fitted regression line through them](../sessions/images/s5-salary-regression-line.png)

**The line runs cleanly through the middle of the points**, and the points sit fairly evenly above and below it. **That is what a good linear fit looks like.**

**What to look for in this plot:**

| If you see | It means |
|---|---|
| Points scattered evenly above and below the line | **A good fit** |
| Points forming a curve around the line | A straight line is the wrong shape |
| The spread widening as x increases | Errors grow with the prediction — a real pattern worth handling |
| A few points very far from the line | Outliers, or something the features do not capture |

> **Always plot a regression, even when the metrics look good.** **R² of 0.899 would look identical whether the relationship is a clean line or a curve the model is cutting through** — and only the picture tells you which.

### Two diagnostic plots worth drawing every time

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))

# Predicted against actual - points should hug the diagonal
ax1.scatter(y_test, y_pred, alpha=.6, edgecolor='k', linewidth=.4)
lims = [y_test.min() - 5000, y_test.max() + 5000]
ax1.plot(lims, lims, 'r--', label='perfect prediction')
ax1.set_xlabel('Actual salary'); ax1.set_ylabel('Predicted salary')
ax1.set_title('Predicted vs actual'); ax1.legend()

# Residuals - the errors, plotted against the prediction
residuals = y_test - y_pred
ax2.scatter(y_pred, residuals, alpha=.6, edgecolor='k', linewidth=.4)
ax2.axhline(0, color='red', linestyle='--')
ax2.set_xlabel('Predicted salary'); ax2.set_ylabel('Error (actual − predicted)')
ax2.set_title('Residuals')

plt.tight_layout()
plt.show()

![Predicted versus actual salaries, and a residual plot](../sessions/images/s5-salary-diagnostics.png)

**Read them together:**

| Plot | What good looks like | What a problem looks like |
|---|---|---|
| **Predicted vs actual** | Points hugging the red diagonal | A curve, or points drifting off at one end |
| **Residuals** | A **shapeless cloud** around zero | A curve, a funnel, or any visible pattern |

> **The residual plot is the more sensitive of the two.** A pattern in the residuals means the model is missing something systematic — **and it will show a pattern long before R² drops enough to worry you.**

# 3. Use case 2 — Advertising spend and sales

**One feature was enough to learn the mechanics. Real problems have several** — and that changes what you can read out of the model.

**The question:** given what was spent on TV, radio and newspaper advertising, predict the resulting sales.

## Steps 1–2 — Load and explore

In [ ]:
dataset_url = "https://raw.githubusercontent.com/tech4alltraining/aiml/refs/heads/main/datasets/regression/advertising.csv"

df = pd.read_csv(dataset_url)
print(df.shape)
df.head()

**Output:**

```text
(200, 4)

      TV  Radio  Newspaper  Sales
0  230.1   37.8       69.2   22.1
1   44.5   39.3       45.1   10.4
2   17.2   45.9       69.3   12.0
```

**Three features and one target.** Each row is one advertising campaign: what was spent on each channel, and what sales followed.

In [ ]:
print("missing:", df.isnull().sum().sum())
print("duplicates:", df.duplicated().sum())

**Output:**

```text
missing: 0
duplicates: 0
```

**Nothing to clean.** That is unusual for real data, and it lets us concentrate on the modelling.

### Correlation — now genuinely useful

**With one feature, correlation told you whether to bother. With three, it starts ranking them.**

In [ ]:
df.corr()['Sales'].sort_values(ascending=False)

**Output:**

```text
Sales        1.0000
TV           0.9012
Radio        0.3496
Newspaper    0.1580
```

**Read that ranking:**

- **TV at 0.90** — a very strong relationship with sales
- **Radio at 0.35** — moderate
- **Newspaper at 0.16** — weak, close to nothing

> **This is a prediction about what the model will find**, and it is worth writing down *before* you train, so you can check whether the model agrees.

### Seeing the three relationships

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, channel in zip(axes, ['TV', 'Radio', 'Newspaper']):
    ax.scatter(df[channel], df['Sales'], alpha=.6, edgecolor='k', linewidth=.3)
    ax.set_title(f"{channel}   (r = {df[channel].corr(df['Sales']):.3f})")
    ax.set_xlabel(f'{channel} spend')
axes[0].set_ylabel('Sales')
plt.tight_layout()
plt.show()

![Three scatter plots: TV, Radio and Newspaper spend against Sales](../sessions/images/s5-advertising-channels.png)

**The three panels show something the correlation numbers cannot:**

- **TV** — a clear upward band, but it **fans out** at higher spend. High TV budgets give less predictable returns.
- **Radio** — a looser upward trend, more scattered throughout.
- **Newspaper** — close to a shapeless cloud, which is what a correlation of 0.16 looks like.

> **That fan shape in the TV panel is exactly why you plot.** No single correlation number contains it, and it would change how you advise a marketing team.

## Steps 3–5 — Split, train, predict

**No preprocessing is needed, so we go straight to the split.**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = df.drop('Sales', axis=1)     # all three channels
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

**Output:**

```text
(160, 3) (40, 3)
```

**Notice `X_train` is now `(160, 3)` rather than `(298, 1)`.** **Three columns instead of one — and not a single line of the modelling code had to change.**

## Step 6 — Evaluate

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  {mae:.4f}")
print(f"MSE  {mse:.4f}")
print(f"RMSE {rmse:.4f}")
print(f"R2   {r2:.4f}")

**Output:**

```text
MAE  1.2748
MSE  2.9078
RMSE 1.7052
R2   0.9059
```

**Sales are measured in thousands of units, so RMSE 1.71 means: typically off by about 1,710 units.**

> **R² of 0.906 with three features, against 0.899 with one in use case 1.** **Different datasets, so the two are not directly comparable** — but both are strong fits.

## Step 7 — Reading three coefficients

**This is what use case 1 could not show you.**

In [ ]:
for name, coef in zip(X.columns, model.coef_):
    print(f"{name:<12}{coef:+.6f}")
print(f"{'intercept':<12}{model.intercept_:+.6f}")

**Output:**

```text
TV          +0.054509
Radio       +0.100945
Newspaper   +0.004337
intercept   +4.714126
```

**The learned equation is:**

```text
Sales = 0.0545 × TV + 0.1009 × Radio + 0.0043 × Newspaper + 4.71
```

**Each coefficient says: how much does Sales change when this channel goes up by one unit, holding the others fixed?**

| Channel | Coefficient | Read as |
|---|---|---|
| **Radio** | +0.1009 | **The strongest per unit spent** |
| **TV** | +0.0545 | About half radio's effect per unit |
| **Newspaper** | +0.0043 | **Essentially nothing** |

> **Radio is worth roughly twice TV per unit of spend.** **That is a budget recommendation, not just a number** — and it is the kind of statement only a linear model hands you directly.

### ⚠️ The correlation and the coefficient disagree — and both are right

**Look back at the correlations:**

```text
correlation:   TV 0.90   >   Radio 0.35   >   Newspaper 0.16
coefficient:   Radio 0.10 >   TV 0.05     >   Newspaper 0.004
```

**TV has the higher correlation, but radio has the higher coefficient. How?**

> **They answer different questions.**
>
> **Correlation** asks: *as TV spend rises, does Sales rise?* TV budgets are large, so TV spend explains a lot of the total variation — hence 0.90.
>
> **The coefficient** asks: *if I spend one more unit on TV, what happens?* **Per unit spent, radio moves sales harder.**
>
> **Both are true.** TV explains more of what happened; radio gives more per rupee. **The first is a description of the past; the second is advice about the next rupee.**

> ⚠️ **Coefficients are only comparable like this when the features are on similar scales.** Here all three are spend in the same units, so the comparison is fair. **If one were in rupees and another in lakhs, you would have to scale first** — use case 3 shows exactly that.

## Step 8 — Should we drop Newspaper?

**Its coefficient is 0.004 and its correlation is 0.16. It looks useless. Test it rather than assuming.**

In [ ]:
X2 = df[['TV', 'Radio']]                  # drop Newspaper
X2_train, X2_test, y_train, y_test = train_test_split(
    X2, y, test_size=0.2, random_state=42)

model2 = LinearRegression().fit(X2_train, y_train)
y_pred2 = model2.predict(X2_test)

print(f"with Newspaper   : R2 {r2_score(y_test, y_pred):.4f}  RMSE {rmse:.4f}")
print(f"without Newspaper: R2 {r2_score(y_test, y_pred2):.4f}  "
      f"RMSE {np.sqrt(mean_squared_error(y_test, y_pred2)):.4f}")

**Output:**

```text
with Newspaper   : R2 0.9059  RMSE 1.7052
without Newspaper: R2 0.9079  RMSE 1.6872
```

> **Removing a feature made the model slightly *better*.**
>
> **That surprises people, because more information ought to help.** But a feature carrying almost no signal still carries noise, and the model spends a little of its capacity fitting that noise. **Drop it and the model has one less distraction.**

**And a simpler model is better for reasons beyond the score:** fewer columns to collect, fewer to explain, fewer things to go wrong in production.

> **The gain here is small — 0.002 of R².** **Session 8 teaches you how to tell whether a difference that small is real** or just the particular rows that landed in the test set. **For now, note that the honest claim is "no worse, and simpler", not "better".**

# 4. Use case 3 — Used car prices

**The first two datasets were clean and cooperative. This one is neither** — and it is much more like what you will actually be handed.

**The question:** given a used car's age, mileage, engine and other details, predict its selling price.

**What is different here:**

| | Use cases 1 & 2 | Use case 3 |
|---|---|---|
| Rows | 200–375 | **15,411** |
| Text columns | None | **Six** |
| Target | Well behaved | **Heavily skewed** |
| Result | Good fit | **Not good enough — and we deal with that honestly** |

## Step 1 — Load and explore

In [ ]:
dataset_url = "https://raw.githubusercontent.com/tech4alltraining/aiml/refs/heads/main/datasets/regression/cardekho_dataset.csv"

df = pd.read_csv(dataset_url)
print(df.shape)
print(df.columns.tolist())

**Output:**

```text
(15411, 14)
['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven',
 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine',
 'max_power', 'seats', 'selling_price']
```

In [ ]:
print("missing:", df.isnull().sum().sum())
print("duplicates:", df.duplicated().sum())
print(df['selling_price'].describe())

**Output:**

```text
missing: 0
duplicates: 0

mean     7.749713e+05
50%      5.560000e+05
max      3.950000e+07
```

> **Mean 774,971 against a median of 556,000, and a maximum of 39.5 million.** **The target is heavily right-skewed** — most cars are ordinary and a few are extremely expensive. **Remember this; it comes back at step 6.**

### Looking at the text columns

In [ ]:
for col in df.select_dtypes('object').columns:
    print(f"{col:<22}{df[col].nunique():>5} distinct")

**Output:**

```text
car_name                121 distinct
brand                    32 distinct
model                   120 distinct
seller_type               3 distinct
fuel_type                 5 distinct
transmission_type         2 distinct
```

**Six text columns, and they are not all the same kind of problem.**

## Step 2 — Preprocessing

### Dropping columns that cannot help

In [ ]:
df = df.drop(columns=['Unnamed: 0', 'car_name', 'model'])
print(df.shape)

**Output:**

```text
(15411, 11)
```

**Three columns removed, each for a different reason:**

| Column | Why it goes |
|---|---|
| `Unnamed: 0` | **A row number** left over from how the file was saved. It is an index, not information |
| `car_name` | **121 distinct values.** Dummy-encoding it would add 120 columns |
| `model` | **120 distinct values**, and it largely repeats `car_name` |

> **`brand` is kept at 32 categories.** That is still a lot, but it is genuine information — a Maruti and a BMW are different markets. **The line between "a useful category" and "too many to encode" is a judgement, and 121 is clearly over it.**

### Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in ['brand', 'seller_type', 'fuel_type', 'transmission_type']:
    df[col] = LabelEncoder().fit_transform(df[col])

print("all numeric:", df.select_dtypes('object').empty)

**Output:**

```text
all numeric: True
```

> ⚠️ **Label Encoding on `brand` implies an order that does not exist** — that brand 20 is somehow more than brand 5. **Session 3 warned about exactly this, and it matters more for linear regression than for a tree**, because a linear model multiplies the code by a weight.
>
> **Dummy variables would be more correct here.** They would also add 31 columns. **We use Label Encoding to keep this walkthrough readable, and note the cost honestly** — one of the tasks at the end asks you to try it the other way and measure the difference.

## Step 3 — Split and scale

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.drop('selling_price', axis=1)
y = df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler().fit(X_train)        # FIT on train only
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train.shape, X_test.shape)

**Output:**

```text
(12328, 10) (3083, 10)
```

**Ten features, 12,328 training rows.**

### ⚠️ Does scaling actually change anything here?

**Test it rather than assuming.**

In [ ]:
model_scaled = LinearRegression().fit(X_train_scaled, y_train)
pred_scaled = model_scaled.predict(X_test_scaled)

model_raw = LinearRegression().fit(X_train, y_train)
pred_raw = model_raw.predict(X_test)

print(f"with scaling   : R2 {r2_score(y_test, pred_scaled):.4f}")
print(f"without scaling: R2 {r2_score(y_test, pred_raw):.4f}")

**Output:**

```text
with scaling   : R2 0.6639
without scaling: R2 0.6639
```

> **Identical, to four decimal places.**
>
> **Linear regression does not need scaling to fit well.** It simply adjusts each coefficient to suit that column's units.
>
> **So why scale at all?** **Because it makes the coefficients comparable with one another** — which is the whole point of step 5 below. **Scaling changes what you can read, not how well the model fits.**

## Step 4 — Train and evaluate

In [ ]:
model = LinearRegression().fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
print(f"MAE  {mean_absolute_error(y_test, y_pred):>12,.0f}")
print(f"RMSE {np.sqrt(mse):>12,.0f}")
print(f"R2   {r2_score(y_test, y_pred):>12.4f}")

**Output:**

```text
MAE       278,522
RMSE      503,024
R2         0.6639
```

### ⚠️ Read this honestly

**R² of 0.66 sounds acceptable. It is not.**

In [ ]:
print(f"median car price : {y.median():>12,.0f}")
print(f"RMSE             : {np.sqrt(mse):>12,.0f}")
print(f"RMSE as a share of the median price: {np.sqrt(mse)/y.median():.0%}")

**Output:**

```text
median car price :      556,000
RMSE             :      503,024
RMSE as a share of the median price: 90%
```

> **The typical error is 90% of the typical car's price.**
>
> **A model that says "this car is worth ₹556,000, give or take ₹503,000" is useless to a buyer or a seller.**
>
> **R² of 0.66 hid that completely.** R² is a ratio with no units, so it cannot tell you whether the remaining error matters. **Only putting RMSE next to the actual scale of the target reveals it.**

**This is the most important lesson in Part A: always compare your error against the size of the thing you are predicting.**

## Step 5 — Reading the coefficients

**Now that the features are scaled, the coefficients are comparable.**

In [ ]:
for name, coef in sorted(zip(X.columns, model.coef_), key=lambda t: -abs(t[1])):
    print(f"{name:<20}{coef:>14,.0f}")

**Output:**

```text
max_power                  641,806
vehicle_age               -181,644
mileage                     64,377
km_driven                  -50,734
engine                      50,421
transmission_type          -44,793
brand                       11,378
seats                        7,003
fuel_type                   -6,147
seller_type                -2,047
```

**These tell a story that matches common sense:**

| Feature | Sign | Reading |
|---|---|---|
| **`max_power`** | **+641,806** | **By far the strongest.** More powerful cars cost much more |
| **`vehicle_age`** | **−181,644** | **Negative** — older cars are worth less. Exactly as expected |
| `km_driven` | −50,734 | Negative — more mileage, lower price |
| `brand` | +11,378 | **Small, and meaningless anyway** — the label codes are arbitrary |

> **The `brand` coefficient is a good example of why Label Encoding on an unordered category is unsatisfying.** The model has fitted a straight line through arbitrary brand numbers. **The number is not wrong arithmetically; it just does not mean anything.**

## Step 6 — Improving it: the skewed target

**Step 1 noticed the target was heavily skewed. That is the problem worth attacking.**

**Linear regression assumes errors are roughly even across the range.** With prices from 100,000 to 39.5 million, they are not — **the model is being pulled around by a handful of very expensive cars.**

**A standard fix is to model the *logarithm* of the price instead.** Taking logs compresses the long tail, so the expensive cars stop dominating.

In [ ]:
y_log = np.log1p(y)          # log(1 + price), so a price of 0 is safe

X_train, X_test, y_log_train, y_log_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42)

model_log = LinearRegression().fit(X_train, y_log_train)
pred_log = model_log.predict(X_test)

print(f"R2 on the original target : 0.6639")
print(f"R2 on the log target      : {r2_score(y_log_test, pred_log):.4f}")

**Output:**

```text
R2 on the original target : 0.6639
R2 on the log target      : 0.8634
```

> **R² rises from 0.66 to 0.86** — a large improvement from one line of preprocessing, and **more than any change of model would have given here.**

> ⚠️ **Two honest caveats.**
>
> **First, the two R² values are not directly comparable** — one is measured on prices, the other on log-prices. **To compare fairly you must convert the predictions back with `np.expm1()` and recompute RMSE in rupees.**
>
> **Second, this did not make the model good — it made it better.** The remaining error is still substantial. **A real used-car model would need features this dataset does not have: condition, service history, accident record, and location.**

### Seeing the difference

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))

ax1.scatter(y_test, y_pred, alpha=.25, s=12)
ax1.plot([0, 4_000_000], [0, 4_000_000], 'r--', label='perfect prediction')
ax1.set_xlim(0, 4_000_000); ax1.set_ylim(0, 4_000_000)
ax1.set_xlabel('Actual price'); ax1.set_ylabel('Predicted price')
ax1.set_title('Original target'); ax1.legend()

ax2.scatter(y_log_test, pred_log, alpha=.25, s=12, color='seagreen')
ax2.set_xlabel('Actual log(price)'); ax2.set_ylabel('Predicted log(price)')
ax2.set_title('Log-transformed target')

plt.tight_layout()
plt.show()

![Predicted versus actual car prices, before and after log-transforming the target](../sessions/images/s5-car-log-transform.png)

**The left panel is what an unusable model looks like.** The points form a wide, shapeless spread rather than following the diagonal — and notice the model rarely predicts anything above about 1.5 million, however expensive the car really was. **It has learned to play safe near the middle.**

**The right panel is visibly tighter.** Taking logs compressed the long tail, so the expensive cars stopped dominating and the model could spread its predictions properly.

**That last point is the honest conclusion: sometimes the limit is the data, not the model.** **No algorithm invents information that was never collected.**

# 5. What the three use cases showed

| | Use case 1 — Salary | Use case 2 — Advertising | Use case 3 — Cars |
|---|---|---|---|
| Rows / features | 373 / 1 | 200 / 3 | **15,411 / 10** |
| Missing values | 2, dropped | None | None |
| Duplicates | **219, deliberately kept** | None | None |
| Encoding | Not needed | Not needed | **4 columns** |
| Scaling | Not needed | Not needed | **For readable coefficients only** |
| R² | 0.899 | 0.906 | **0.664** |
| RMSE in context | ₹15,551 on ~₹95,000 | 1.7 on ~14 | **₹503,024 on ~₹556,000** |
| Verdict | Good | Good | **Not good enough** |

**Four things worth carrying forward:**

1. **The code barely changed.** One feature or ten, 200 rows or 15,000 — `fit`, `predict`, and the same four metrics. **The workflow is the skill; the dataset decides the judgements.**

2. **Not every duplicate is an error.** Use case 1 kept 219 of them, because two people with the same experience and salary are two people.

3. **A metric with no units can hide a bad model.** R² of 0.66 sounded acceptable until RMSE was placed next to the median price.

4. **Sometimes the limit is the data.** Use case 3 improved a lot from a log transform and still was not good — because the columns that would explain a used car's price were never collected.

# ✅ Before you move on

- [ ] I can tell regression from classification by looking at the target column
- [ ] I know why `stratify` is not used in a regression split
- [ ] I can explain why the **sum** of the errors is a useless summary
- [ ] I know MAE averages absolute errors and MSE averages squared ones
- [ ] I know MSE's units are the target **squared**, and never quote it to a person
- [ ] **I report RMSE, because it is in the target's own units**
- [ ] I know R² compares the model against always guessing the average
- [ ] I know R² can be negative, and that 1.0 on a test set is a warning
- [ ] I compare RMSE against the target's median before calling a model good
- [ ] I can read a coefficient out loud as a plain-English sentence
- [ ] I know correlation and coefficient answer different questions
- [ ] I plot predicted-vs-actual and residuals, not just the metrics
- [ ] **I know that sometimes the limit is the data, not the model**

## More practice

| Where | What |
|---|---|
| [Notebook](../notebooks/session-05-regression.ipynb) | All three use cases, runnable |
| [Session 5B — Classification](../sessions/session-05b-classification.md) | The other half of supervised learning |
| [Exercises & assignments](../exercises-assignments.md) | Longer graded work |

---

# Next steps

**The 20 MCQs and 20 preprocessing tasks are in the guide:**
[Session 3 guide](../sessions/session-05-regression.md)

| | |
|---|---|
| **Previous** | [Session 2 — NumPy, Pandas & Visualisation](../sessions/session-02-numpy-pandas.md) |
| **Next** | [Session 4 — Introduction to AI & ML](../sessions/session-04-intro-ml-ai.md) |
| **Stuck?** | [Troubleshooting](../troubleshooting.md) |